In [23]:
!pip install boto3

In [6]:
# koneksi dengan data LAke jalur API
import boto3

s3 = boto3.client(
    's3',
    endpoint_url='http://minio1:9001', # atau http://localhost:9000 jika dari luar container
    aws_access_key_id='admin',
    aws_secret_access_key='secretsecret'
)

In [14]:
# Tampilkan daftar bucker
response = s3.list_buckets()
for bucket in response.get('Buckets', []):
    print(f"- {bucket['Name']}")

- bronze
- coba
- gold
- silver


In [21]:
# buat bucket dengan nama bronze, silver, gold 
s3.create_bucket(Bucket='bronze')

BucketAlreadyOwnedByYou: An error occurred (BucketAlreadyOwnedByYou) when calling the CreateBucket operation: Your previous request to create the named bucket succeeded and you already own it.

In [22]:
# upload sales.csv ke bucket bronze

In [24]:
!pip install pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 2.5 MB/s  0:00:04 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 2.3 MB/s  0:00:07m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━ 1/2 [pandas]^C
ERROR: Operation cancelled by user
   ━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━ 1/2 [pandas]


In [31]:
# bersihkan data sales.csv lalu simpan bucket silver dengan nama sales_cleaned.csv
import pandas as pd
df = pd.read_csv('sales.csv')

print("--- Data Asli (Raw) ---")
display(df)

# ==========================================
# 2. PROSES DATA CLEANING (Pembersihan Data)
# ==========================================
# Bersihkan spasi berlebih pada kolom teks (store & produk) jika ada
df['store'] = df['store'].astype(str).str.strip()
df['produk'] = df['produk'].astype(str).str.strip()

# Hapus baris di mana harga atau jumlah bernilai 0 atau negatif
df_clean = df[(df['harga'] > 0) & (df['jumlah'] > 0)]

print("\n--- Data Setelah Dibersihkan ---")
display(df_clean)

# ==========================================
# 3. KONEKSI KE MINIO & SIMPAN KE BUCKET SILVER
# ==========================================
s3 = boto3.client(
    's3',
    endpoint_url='http://minio1:9000',
    aws_access_key_id='admin',
    aws_secret_access_key='secretsecret',
)

bucket_name = 'silver'

# Pastikan bucket silver ada (meskipun di list sudah ada)
try:
  s3.head_bucket(Bucket=bucket_name)
except Exception:
  s3.create_bucket(Bucket=bucket_name)

# Konversi DataFrame bersih ke CSV string, lalu upload ke MinIO
csv_data = df_clean.to_csv(index=False)
s3.put_object(
    Bucket=bucket_name, Key='sales_cleaned.csv', Body=csv_data.encode('utf-8')
)

print(
    f"\nSukses! File 'sales_cleaned.csv' berhasil dibersihkan dan disimpan ke"
    f" bucket '{bucket_name}'."
)

--- Data Asli (Raw) ---


,id,store,produk,harga,jumlah
0,1,s1,p1,1000,2
1,2,s2,p2,2000,3
2,3,s3,p3,0,0
3,4,s2,p4,1400,6



--- Data Setelah Dibersihkan ---


,id,store,produk,harga,jumlah
0,1,s1,p1,1000,2
1,2,s2,p2,2000,3
3,4,s2,p4,1400,6



Sukses! File 'sales_cleaned.csv' berhasil dibersihkan dan disimpan ke bucket 'silver'.


In [32]:
# aggregate data sales_cleanded.csv lalu simpan ke bucket gold dan tampilkan
# 1. Hitung total per store dari data yang sudah ada (df_clean)
df_clean['total_penjualan'] = df_clean['harga'] * df_clean['jumlah']

# 2. Agregasi data (kolom: id, store, total)
df_gold = (
    df_clean.groupby('store')['total_penjualan'].sum().reset_index(name='total')
)
df_gold.insert(0, 'id', range(1, len(df_gold) + 1))

# Tampilkan hasil
print('--- Hasil Agregasi (Bucket Gold) ---')
display(df_gold)

# 3. Simpan ke bucket 'gold' di MinIO
bucket_gold = 'gold'
try:
  s3.head_bucket(Bucket=bucket_gold)
except Exception:
  s3.create_bucket(Bucket=bucket_gold)

csv_gold_data = df_gold.to_csv(index=False)
s3.put_object(
    Bucket=bucket_gold,
    Key='sales_aggregated.csv',
    Body=csv_gold_data.encode('utf-8'),
)

print(f"\nSukses! Berhasil disimpan ke bucket '{bucket_gold}'.")

--- Hasil Agregasi (Bucket Gold) ---


,id,store,total
0,1,s1,2000
1,2,s2,14400



Sukses! Berhasil disimpan ke bucket 'gold'.
